<div style="background: linear-gradient(135deg,#00352E 0%,#00695C 100%);
            padding:40px 36px 32px 36px;
            border-radius:12px;
            color:white;
            font-family:Arial,sans-serif">

<div style="font-size:13px; letter-spacing:2px; opacity:.75; margin-bottom:8px">
UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026
</div>

<h1 style="margin:0 0 6px 0; font-size:28px; font-weight:700; line-height:1.2">
Preprocessing (Silver) — SPARCS
</h1>

<h2 style="margin:0 0 20px 0; font-size:17px; font-weight:400; opacity:.9">
Limpieza y estandarización del dataset de Nueva York, 2021
</h2>

<hr style="border:none; border-top:1px solid rgba(255,255,255,.3); margin:0 0 18px 0"/>

<div style="display:flex; gap:40px; font-size:13px; opacity:.85">

<div>
<b>Autores:</b><br>
Daniel Vallejo Vecil<br>
Juliana Muñoz González
</div>

<div>
<b>Director:</b><br>
Julián F. Sánchez López
</div>

<div>
<b>Capa construida:</b><br>
2_silver
</div>

</div>

</div>

<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
1. Objetivo
</h2>

</div>

<div style="background:#E8F5F1;
border-left:6px solid #00695C;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Este notebook construye la capa <b>Silver</b> a partir del archivo Bronze de SPARCS, aplicando las
decisiones documentadas en el EDA (<code>01_EDA_SPARCS.ipynb</code>, ver
<code>01_SintesisVariables.csv</code>):

<ul>
<li>Eliminar identificadores puros y variables constantes.</li>
<li>Limpiar y convertir <code>Length of Stay</code> a numérico (incluye el caso <code>"120 +"</code>
con espacio).</li>
<li>Tratar los nulos <b>informativos</b> de <code>CCSR Procedure Code/Description</code> y
<code>Payment Typology 2/3</code> como una categoría explícita, no como dato faltante a imputar.</li>
<li>Eliminar <code>Birth Weight</code> por su alto porcentaje de nulos (90.2%).</li>
<li>Eliminar las filas con nulos residuales de baja proporción (&lt;2%) en variables geográficas.</li>
</ul>


</div>

<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
2. Configuración y carga
</h2>

</div>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"
BRONZE_PATH = DATA_PATH / "1_bronze"
SILVER_PATH = DATA_PATH / "2_silver"
SILVER_PATH.mkdir(parents=True, exist_ok=True)

TABLAS_PATH = PROJECT_ROOT / "tablas"
TABLAS_PATH.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NUM = "02"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


RAW_FILE = BRONZE_PATH / "Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2021_20260906.csv"
df = pd.read_csv(RAW_FILE, low_memory=False)

print(f"Filas cargadas (Bronze):    {df.shape[0]:,}")
print(f"Columnas cargadas (Bronze): {df.shape[1]}")


Filas cargadas (Bronze):    2,135,260
Columnas cargadas (Bronze): 33


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
3. Eliminar duplicados
</h2>

</div>

In [2]:
n_antes = df.shape[0]
df = df.drop_duplicates()
n_despues = df.shape[0]

print(f"Duplicados eliminados: {n_antes - n_despues:,}")
print(f"Filas restantes: {n_despues:,}")


Duplicados eliminados: 4,097
Filas restantes: 2,131,163


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
4. Eliminar identificadoreS, variables constantes y de pagos
</h2>

</div>

In [3]:
COLUMNAS_ELIMINAR = [
    "Operating Certificate Number",  # identificador puro
    "Permanent Facility Id",         # identificador puro
    "Facility Name",                 # identificador puro (alta cardinalidad, sin valor predictivo)
    "Discharge Year",                # constante (2021) en todo el dataset
    "Total Charges",
    "Total Costs",
    "Patient Disposition"
]

columnas_presentes = [c for c in COLUMNAS_ELIMINAR if c in df.columns]
df = df.drop(columns=columnas_presentes)

print(f"Columnas eliminadas: {columnas_presentes}")
print(f"Columnas restantes: {df.shape[1]}")


Columnas eliminadas: ['Operating Certificate Number', 'Permanent Facility Id', 'Facility Name', 'Discharge Year', 'Total Charges', 'Total Costs', 'Patient Disposition']
Columnas restantes: 26


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
5. Limpieza de la variable objetivo: Length of Stay
</h2>

</div>

In [4]:
los_raw = df["Length of Stay"].astype(str).str.strip()

# El valor agregado real en el archivo es "120 +" (con espacio) para estancias >= 120 días
df["Length of Stay"] = pd.to_numeric(
    los_raw.str.replace(r"120\s*\+", "120", regex=True),
    errors="coerce"
)

print(f"Valores no convertibles a número (deberían ser 0): {df['Length of Stay'].isna().sum()}")
print()
print(df["Length of Stay"].describe().round(2))


Valores no convertibles a número (deberían ser 0): 0

count    2131163.00
mean           5.75
std            8.42
min            1.00
25%            2.00
50%            3.00
75%            6.00
max          120.00
Name: Length of Stay, dtype: float64


In [5]:
# Filas donde la limpieza no pudo convertir el valor: se eliminan (dato objetivo inválido)
n_antes = df.shape[0]
df = df[df["Length of Stay"].notna()].copy()
n_despues = df.shape[0]

print(f"Filas eliminadas por Length of Stay no numérica: {n_antes - n_despues}")
print(f"Filas restantes: {n_despues:,}")


Filas eliminadas por Length of Stay no numérica: 0
Filas restantes: 2,131,163


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
6. Nulos informativos: tratarlos como categoría explícita
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


En estas variables, el valor nulo <b>no es un dato faltante por error</b>: significa que la atención
no tuvo un segundo/tercer pagador, o no tuvo un procedimiento asociado (solo diagnóstico). Imputar con
la moda o eliminar esas filas destruiría información real. Se reemplazan por una categoría explícita.


</div>

In [6]:
NULOS_INFORMATIVOS = {
    "CCSR Procedure Code": "SIN_PROCEDIMIENTO",
    "CCSR Procedure Description": "SIN_PROCEDIMIENTO",
    "Payment Typology 2": "SIN_PAGADOR_2",
    "Payment Typology 3": "SIN_PAGADOR_3",
}

for columna, valor in NULOS_INFORMATIVOS.items():
    if columna not in df.columns:
        continue
    n_nulos = df[columna].isna().sum()
    df[columna] = df[columna].fillna(valor)
    print(f"{columna}: {n_nulos:,} nulos reemplazados por '{valor}'")


CCSR Procedure Code: 580,893 nulos reemplazados por 'SIN_PROCEDIMIENTO'
CCSR Procedure Description: 580,893 nulos reemplazados por 'SIN_PROCEDIMIENTO'
Payment Typology 2: 1,094,489 nulos reemplazados por 'SIN_PAGADOR_2'
Payment Typology 3: 1,800,012 nulos reemplazados por 'SIN_PAGADOR_3'


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
7. Eliminar Birth Weight (90.2% de nulos)
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Se elimina por su altísimo porcentaje de nulos, consistente con el umbral de &gt;90% usado en el
proyecto anterior. Solo aplica a recién nacidos, y esa información ya queda parcialmente capturada en
<code>Type of Admission = "Newborn"</code> y en <code>APR MDC Description</code> (categoría de
neonatos).


</div>

In [7]:
if "Birth Weight" in df.columns:
    df = df.drop(columns=["Birth Weight"])
    print("Birth Weight eliminada (90.2% de nulos)")

print(f"Columnas restantes: {df.shape[1]}")


Birth Weight eliminada (90.2% de nulos)
Columnas restantes: 25


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
8. Nulos residuales de baja proporción: eliminar filas
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Las variables geográficas (<code>Hospital Service Area</code>, <code>Hospital County</code>,
<code>Zip Code - 3 digits</code>) tienen muy pocos nulos (&lt;2%). A diferencia de los nulos
informativos de la sección anterior, aquí sí representan datos faltantes reales — se eliminan las
filas en vez de inventar una categoría, dado el bajo impacto en el tamaño de la muestra.


</div>

In [8]:
COLUMNAS_NULOS_RESIDUALES = ["Hospital Service Area", "Hospital County", "Zip Code - 3 digits"]
columnas_presentes = [c for c in COLUMNAS_NULOS_RESIDUALES if c in df.columns]

n_antes = df.shape[0]
df = df.dropna(subset=columnas_presentes)
n_despues = df.shape[0]

print(f"Filas eliminadas por nulos residuales: {n_antes - n_despues:,} ({(n_antes-n_despues)/n_antes*100:.2f}%)")
print(f"Filas restantes: {n_despues:,}")


Filas eliminadas por nulos residuales: 40,244 (1.89%)
Filas restantes: 2,090,919


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
9. Verificación final de nulos
</h2>

</div>

In [9]:
nulos_finales = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
print("Nulos restantes por columna:")
display(nulos_finales[nulos_finales > 0].to_frame("% nulos") if (nulos_finales > 0).any()
        else "Sin valores nulos restantes.")

guardar_tabla(nulos_finales.to_frame("pct_nulos"), "NulosResidualesSilver")


Nulos restantes por columna:


,% nulos
APR Risk of Mortality,0.03
APR Severity of Illness Description,0.03


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\02_NulosResidualesSilver.csv


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
10. Resumen del proceso de limpieza
</h2>

</div>

In [10]:
resumen_limpieza = pd.DataFrame([
    {"etapa": "Bronze (original)", "filas": 2_135_260, "columnas": 33},
    {"etapa": "Silver (final)", "filas": df.shape[0], "columnas": df.shape[1]},
])
resumen_limpieza["filas_eliminadas"] = resumen_limpieza["filas"].iloc[0] - resumen_limpieza["filas"]
resumen_limpieza["pct_filas_conservadas"] = (resumen_limpieza["filas"] / resumen_limpieza["filas"].iloc[0] * 100).round(2)

display(resumen_limpieza)
guardar_tabla(resumen_limpieza, "ResumenLimpieza")


,etapa,filas,columnas,filas_eliminadas,pct_filas_conservadas
0,Bronze (original),2135260,33,0,100.00
1,Silver (final),2090919,25,44341,97.92


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\02_ResumenLimpieza.csv


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
11. Exportación de la capa Silver
</h2>

</div>

In [11]:
silver_file = SILVER_PATH / "sparcs_silver.parquet"
df.to_parquet(silver_file, index=False)

print("=" * 60)
print("Dataset Silver exportado correctamente")
print("=" * 60)
print(f"Ubicación : {silver_file.resolve()}")
print(f"Filas     : {df.shape[0]:,}")
print(f"Columnas  : {df.shape[1]}")
print(f"Columnas finales: {df.columns.tolist()}")


Dataset Silver exportado correctamente
Ubicación : C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\data\2_silver\sparcs_silver.parquet
Filas     : 2,090,919
Columnas  : 25
Columnas finales: ['Hospital Service Area', 'Hospital County', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Length of Stay', 'Type of Admission', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description', 'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Code', 'APR DRG Description', 'APR MDC Code', 'APR MDC Description', 'APR Severity of Illness Code', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3', 'Emergency Department Indicator']


<div style="background:#00695C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
12. Conclusiones
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


La capa Silver queda lista para <code>03_FeatureEngineering.ipynb</code>, con las siguientes
decisiones aplicadas: eliminación de 4 identificadores/constantes, limpieza de la variable objetivo
(incluyendo el caso <code>"120 +"</code>), tratamiento de nulos informativos como categoría explícita
en 4 variables, eliminación de <code>Birth Weight</code>, y eliminación de filas con nulos residuales
geográficos de bajo impacto.


</div>